In [9]:
# Import libraries

In [ ]:
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Draw, PandasTools
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
from rdchiral import template_extractor
from tqdm import tqdm
import pandas as pd

# Read the data

In [ ]:
df = pd.read_csv("/home/bilal/Retroproject/Retrosynthesis-Prediction-master/notebooks/uspto50k/data_processed.csv")

In [ ]:
# Drop unwanted rows

In [ ]:
df.drop(['id', 'class','reactant','reaction','reaction_template' ], axis = 1, inplace = True)

In [ ]:
df.shape

In [8]:
# Remove Product Redundancy

In [ ]:
import pandas as pd
from rdkit.Chem import DataStructs

# Initialize lists to store unique sequences and labels
unique_sequences = []
unique_labels = []


# Iterate through rows of the processed dataset
for index, row in df.iterrows():
    sequence = row['product']
    labels = row['label']
    #morgan_fp = ','.join(str(int(bit)) for bit in row['Morgan_Fingerprint'].ToBitString())

    # Check if the sequence is unique
    if sequence not in unique_sequences:
        unique_sequences.append(sequence)
        unique_labels.append(labels)
        #unique_morgan_fp.append(morgan_fp)

# Create a new DataFrame with unique sequences and labels
unique_df2 = pd.DataFrame({'unique_product': unique_sequences, 'unique_label': unique_labels})

# Save the DataFrame with unique sequences and labels to a new CSV file
unique_df2.to_csv('unique_processed_prod3.csv', index=False)

In [ ]:
data3 = pd.read_csv('unique_processed_prod3.csv')

In [ ]:
# Functional Group Interchange Data Augmentation

In [ ]:
from rdkit import Chem
import numpy as np
import pandas as pd

def get_random_functional_group():
    # Example implementation with a more diverse set of functional groups
    functional_groups = ['[CH3]', '[OH]', '[NH2]', '[C@H](F)Cl', '[C@@H](Br)N']
    return np.random.choice(functional_groups)

def generate_augmented_smiles(smiles, num_variations=2):
    mol = Chem.MolFromSmiles(smiles)
    
    if mol and mol.GetNumAtoms() > 1:
        augmented_smiles_list = []
        
        for _ in range(num_variations):
            current_mol = Chem.RWMol(mol)
            
            # Perform functional group interchange augmentation
            for _ in range(2):  # You can adjust the number of interchanges as needed
                if current_mol.GetNumBonds() == 0:
                    break
                
                bond_index = np.random.randint(0, current_mol.GetNumBonds())
                bond = current_mol.GetBondWithIdx(bond_index)
                atom1_index, atom2_index = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
                
                try:
                    functional_group = Chem.MolFragmentToSmiles(current_mol, atom1_index, atom2_index, kekuleSmiles=True)
                except:
                    continue
                
                new_functional_group = get_random_functional_group()
                
                current_mol.RemoveBond(atom1_index, atom2_index)
                current_mol = Chem.CombineMols(current_mol, Chem.MolFromSmiles(new_functional_group))
            
            augmented_smiles_list.append(Chem.MolToSmiles(current_mol))
        
        return augmented_smiles_list

    else:
        return [smiles]

# Apply the augmentation function to each row in the dataset
data3['augmented_prods'] = data3['unique_product'].apply(generate_augmented_smiles)

# Explode the list of augmented SMILES strings into multiple rows
data3_augmented = data3.explode('augmented_prods').reset_index(drop=True)




In [ ]:
data3_augmented.to_csv('processed_data_aug', index = False)

In [ ]:
data = pd.read_csv('processed_data_aug')

In [7]:
# Maximium unique-product length

In [ ]:
# Index of the longest SMILES string
longest_smiles = max(data["augmented_prods"], key=len)
longest_smiles_index = data[data["augmented_prods"] == longest_smiles].index.tolist()
print(f"Longest SMILES: {longest_smiles}")
print(f"Contains {len(longest_smiles)} characters, index in dataframe: {longest_smiles_index[0]}.")
smiles_maxlen = len(longest_smiles)

In [6]:
# Minimium unique-product length

In [ ]:
# Index of the longest SMILES string
smallest_smiles = min(data["augmented_prods"], key=len)
smallest_smiles_index = data[data["augmented_prods"] == smallest_smiles].index.tolist()
print(f"smallest SMILES: {smallest_smiles}")
print(f"Contains {len(smallest_smiles)} characters, index in dataframe: {smallest_smiles_index[0]}.")
smiles_minlen = len(smallest_smiles)

In [5]:
# Plot to check the Sequence lengths

In [ ]:
import matplotlib.pyplot as plt

# Calculate the lengths of each sequence and store them in a new column
data['Sequence_Length'] = data['augmented_prods'].apply(len)

# Check the minimum and maximum sequence lengths
min_length = data['Sequence_Length'].min()
max_length = data['Sequence_Length'].max()

print(f"Minimum Sequence Length: {min_length}")
print(f"Maximum Sequence Length: {max_length}")

# Create a horizontal bar plot to visualize the sequence lengths
plt.figure(figsize=(8, 8))  # Adjust the figure size as needed
plt.barh(data.index, data['Sequence_Length'], color='blue')
plt.xlabel('Sequence Length')
plt.ylabel('Sequence Index')
plt.title('Sequence Lengths (Longest at the Bottom)')
plt.grid(True)
plt.show()


In [4]:
# Trim the unique-products sequences to max-length 300

In [ ]:
trimmed_products = []
#trimmed_reactants = []

max_sequence_length = 300  # Maximum allowed sequence length

for index, row in data.iterrows():
    pp_product = row['augmented_prods']
    #reactant = row['reactants']
  

    # Trim sequences to maximum length of 300
    if len(pp_product) > max_sequence_length:
        
        pp_product = pp_product[:max_sequence_length]
#     if len(reactant) > max_sequence_length:
#         reactant = reactant[:max_sequence_length]
    
    
    # Skip sequences with length less than 1
    if len(pp_product) < 1:
        #or len(reactant) < 1:
        continue
    
    # Append trimmed data to lists
    trimmed_products.append(pp_product)
    #trimmed_reactants.append(reactant)

# Create a new DataFrame with the trimmed data
trimmed_dp = pd.DataFrame({
    'trim_prod': trimmed_products
    #'reactants': trimmed_reactants
})

In [ ]:
trimmed_dp['labels'] = data['unique_label']

In [ ]:
trimmed_dp.head()

In [3]:
# Process the strings in the dataset

In [ ]:
def assess_two_letter_elements(trimmed_dp):


    # Search for unique characters in SMILES strings
    unique_chars = set(trimmed_dp['trim_prod'].apply(list).sum())
    # Get upper and lower case letters only
    upper_chars = []
    lower_chars = []
    for entry in unique_chars:
        if entry.isalpha():
            if entry.isupper():
                upper_chars.append(entry)
            elif entry.islower():
                lower_chars.append(entry)
    print(f"Upper letter characters {sorted(upper_chars)}")
    print(f"Lower letter characters {sorted(lower_chars)}")
    
    periodic_elements = ["Ac","Al","Am","Sb","Ar","As","At","Ba","Bk","Be","Bi","Bh","B","Br","Cd","Ca","Cf","C","Ce","Cs","Cl",
                         "Cr","Co","Cn","Cu","Cm","Ds","Db","Dy", "Es", "Er", "Eu", "Fm", "Fl", "F", "Fr", "Gd", "Ga", "Ge", 
                         "Au","Hs","He","Ho","H","In","I","Ir","Fe","Kr","La","Lr","Pb","Li","Lv","Lu","Mg","Mn","Mt","Md","Hg",
                         "Mo","Mc","Nd","Ne","Np","Ni","Nh","Nb","N","No","Og","Os", "O", "Pd", "P", "Pt", "Pu", "Po", "K", "Pr",
                         "Pm","Pa","Ra","Rn","Re","Rh","Rg","Rb","Ru","Rf","Sm","Sc","Sg","Se","Si","Ag","Na","Sr","S","Ta","Tc",
                         "Te","Ts","Tb","Tl","Th","Tm","Sn","Ti","W","U","V","Xe","Yb","Y","Zn","Zr"]

    # The two_char_elements list contains all two letter elements
    # which can be generated by all possible combination of upper x lower characters
    # and are valid periodic elements.
    two_char_elements = []
    for upper in upper_chars:
        for lower in lower_chars:
            ch = upper + lower
            if ch in periodic_elements:
                two_char_elements.append(ch)

    # This list is then reduced to the subset of two-letter elements
    # that actually appear in the SMILES strings, specific to our data set.
    two_char_elements_smiles = set()
    for char in two_char_elements:
        if trimmed_dp['trim_prod'].str.contains(char).any():
            two_char_elements_smiles.add(char)

    return two_char_elements_smiles, unique_chars

In [ ]:
elements_found = assess_two_letter_elements(trimmed_dp)
print(f"\nTwo letter elements found in the data set: {sorted(elements_found)}")

In [ ]:
replace_dict = {"Cl": "L", "Br": "R", "Si": "X", "Sn": "Z", 'se': 'E'}

In [ ]:
def preprocessing_data(trimmed_dp, replacement):

    # Print warning if the data set has a 'Sc' element
    if trimmed_dp['trim_prod'].str.contains("Sc").any():
        print(
            'Warning: "Sc" element is found in the data set, since the element is rarely found '
            "in the drugs so we are not converting  "
            'it to single letter element, instead considering "S" '
            'and "c" as separate elements. '
        )

    # Create a new column having processed  SMILES
    trimmed_dp["processed_products"] = trimmed_dp["trim_prod"].copy()

    # Replace the two letter elements found with one character
    for pattern, repl in replacement.items():
        trimmed_dp["processed_products"] = trimmed_dp["processed_products"].str.replace(pattern, repl)

    unique_char = set(trimmed_dp['processed_products'].apply(list).sum())
    return trimmed_dp, unique_char

In [ ]:
trimmed_dp, unique_char = preprocessing_data(trimmed_dp, replace_dict)


In [ ]:
aug_data = trimmed_dp[['labels', 'processed_products']]

In [ ]:
aug_data.head(2)

In [ ]:
aug_data.shape

In [2]:
# Convert processed_products into one hot encoding

In [ ]:
#Function defined to create one-hot encoded matrix
def smiles_encoder(smiless, max_len, unique_char):
 
    # create dictionary of the unique char data set
    smi2index = {char: index for index, char in enumerate(unique_char)}
    # one-hot encoding
    # zero padding to max_len
    smiles_matrix = np.zeros((len(unique_char), max_len))
    for index, char in enumerate(smiless):
        smiles_matrix[smi2index[char], index] = 1
    return smiles_matrix

In [ ]:
# Apply the function to the processed_SMILES strings
#df["unique_char_ohe_matrix"] = df["processed_products"].apply(smiles_encoder, max_len=229, unique_char=unique_char)
OHE_pp=[]
for j in aug_data["processed_products"]:
    OHE_pp.append(smiles_encoder(j,300,unique_char=unique_char))
OHE_pp=np.asarray(OHE_pp)
print(OHE_pp[2])

In [ ]:

# # Reshape OHE_pp to have the desired shape (49676, 300, 42)
OHE_pp_reshaped = np.transpose(OHE_pp, (0, 2, 1))

In [1]:
# ECFP fingerprints

In [ ]:
# Initialize a list to store ECFP representations
ecfp_list = []

for index, row in aug_data.iterrows():
    product_smiles = row['processed_products']
    # Convert SMILES to RDKit molecule
    mol = Chem.MolFromSmiles(product_smiles)
    if mol is not None:
        # Generate ECFP fingerprint (change radius and bitInfo as needed)
        ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048, bitInfo=None)
        # Convert ECFP to a list of binary values
        ecfp_bits = list(map(int, ecfp.ToBitString()))
        ecfp_list.append(ecfp_bits)
    else:
        ecfp_list.append([0] * 2048)  # Use zero vector for invalid SMILES

# Create a DataFrame with ECFP representations
ecfp_df3 = pd.DataFrame(ecfp_list, columns=[f'ECFP_{i}' for i in range(2048)])


In [ ]:
import numpy as np

# Assuming ecfp_input is a Keras symbolic input
# Convert it to a NumPy array
ecfp_input_array = np.array(ecfp_df3)

# Define the desired kernel size
kernel_size = 2048

# Pad sequences to a minimum length of kernel_size
padded_ecfp_input = np.zeros((len(ecfp_input_array), kernel_size, 1))
for i, sequence in enumerate(ecfp_input_array):
    if len(sequence) >= kernel_size:
        padded_ecfp_input[i, :, 0] = sequence[:kernel_size]
    else:
        padded_ecfp_input[i, :len(sequence), 0] = sequence


In [ ]:
# Label Processing

In [ ]:

labels_column = aug_data['labels']

# Extract and process the labels
labels = []
for label in labels_column:
    labels.append(label)
#print(labels)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
# Convert encoded labels to one-hot encoded format
onehot_encoded_cllabels = to_categorical(encoded_labels)

In [ ]:
onehot_products = OHE_pp_reshaped
ecfp_products = padded_ecfp_input
labels = onehot_encoded_cllabels

In [ ]:
onehot_products.shape,  ecfp_products.shape,    labels.shape

In [ ]:
# Model Construction

In [ ]:
from keras.layers import Input, Dense, LSTM, Conv1D, MaxPooling1D, Flatten, Bidirectional, PReLU, GRU, BatchNormalization, concatenate, Dropout
from keras.models import Model
from keras.optimizers import Adam
from keras.regularizers import l2
# Define input layers
input_onehot = Input(shape=(300, 42), name='input_onehot')
input_ecfp = Input(shape=(2048, 1), name='input_ecfp')

# One-hot branch with Convolutional and LSTM layers
conv1d_out_onehot = Conv1D(256, kernel_size=5, activation='PReLU')(input_onehot)
maxpool_out_onehot = MaxPooling1D(pool_size=4)(conv1d_out_onehot)
lstm_out_onehot = Bidirectional(LSTM(512))(maxpool_out_onehot)
onehot_dense = BatchNormalization()(lstm_out_onehot)
onehot_dense = Dropout(0.5)(onehot_dense)

# ECFP branch with Convolutional layer
conv1d_out_ecfp = Conv1D(256, kernel_size=7, activation='PReLU')(input_ecfp)
maxpool_out_ecfp = MaxPooling1D(pool_size=4)(conv1d_out_ecfp)
ecfp_lstm_out = Bidirectional(LSTM(512))(maxpool_out_ecfp)
ecfp_dense = BatchNormalization()(ecfp_lstm_out)
ecfp_dense = Dropout(0.5)(ecfp_dense)

# Concatenate the outputs of both branches
merged = concatenate([onehot_dense, ecfp_dense])

# Additional dense layers with L2 regularization
dense_layer1 = Dense(512, activation='PReLU', kernel_regularizer=l2(0.01))(merged)
dense_layer1 = BatchNormalization()(dense_layer1)
dense_layer1 = Dropout(0.5)(dense_layer1)

dense_layer2 = Dense(1024, activation='PReLU', kernel_regularizer=l2(0.01))(dense_layer1)
dense_layer2 = BatchNormalization()(dense_layer2)
dense_layer2 = Dropout(0.5)(dense_layer2)

# Output layer with 11,852 classes and softmax activation
output_layer = Dense(11852, activation='softmax', name='output')(dense_layer2)

# Create the model
model = Model(inputs=[input_onehot, input_ecfp], outputs=output_layer)

# Compile the model
opt = Adam()
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()


In [ ]:
# Split data into train test validation

In [ ]:
from sklearn.model_selection import train_test_split


# Split the data into training (80%) and the rest (20%)

OHE_pp_train, OHE_pp_temp, padded_ecfp_input_train, padded_ecfp_input_temp, onehot_encoded_labels_train, onehot_encoded_labels_temp = train_test_split([onehot_products, ecfp_products], labels, test_size=0.2, random_state=90)

# Now, split the "temp" set into validation (10%) and test (10%)
OHE_pp_validation, OHE_pp_test, padded_ecfp_input_validation, padded_ecfp_input_test, onehot_encoded_labels_validation, onehot_encoded_labels_test = train_test_split([OHE_pp_temp, padded_ecfp_input_temp], onehot_encoded_labels_temp, test_size=0.5, random_state=42)



In [ ]:
# Train the model

In [ ]:
from keras.utils import Sequence
from keras.callbacks import LearningRateScheduler
from keras.callbacks import LearningRateScheduler, ModelCheckpoint, EarlyStopping

from keras.callbacks import ModelCheckpoint
# Define a custom data generator
class DataGenerator(Sequence):
    def __init__(self, onehot_data, ecfp_data, labels, batch_size):
        self.onehot_data = onehot_data
        self.ecfp_data = ecfp_data
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.labels))

    def __len__(self):
        return int(np.ceil(len(self.labels) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        batch_onehot = self.onehot_data[start:end]
        batch_ecfp = self.ecfp_data[start:end]
        batch_labels = self.labels[start:end]

        return [batch_onehot, batch_ecfp], batch_labels

# Create instances of the data generator for training and validation
train_generator = DataGenerator(OHE_pp_train, padded_ecfp_input_train, onehot_encoded_labels_train, batch_size=128)

validation_generator = DataGenerator(OHE_pp_validation, padded_ecfp_input_validation, onehot_encoded_labels_validation, batch_size=256)


# Define the learning rate schedule function
def lr_schedule(epoch):
    if epoch < 10:
        return 0.001
    elif epoch < 20:
        return 0.0001
    else:
        return 0.00001

# Create the learning rate scheduler callback
lr_scheduler = LearningRateScheduler(lr_schedule)

# Define other callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
checkpoint = ModelCheckpoint("CNN-Retro_best_weights.h5", save_best_only=True)

# Fit the model using fit
history = model.fit(
    train_generator,
    epochs=70,
    validation_data=validation_generator,
    callbacks=[early_stop, lr_scheduler, checkpoint]
)


# Save the model architecture to a JSON file
model_json = model.to_json()
with open("CNN-Retro.json", "w") as json_file:
    json_file.write(model_json)

# Save the final trained weights to an HDF5 file
model.save_weights("CNN-Retro_final_weights.h5")

In [ ]:
# Plotting

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


In [ ]:
# Test the model

In [ ]:
from tensorflow.keras.models import model_from_json
from tensorflow.keras.optimizers import Adam
import numpy as np

# 1. Load model architecture from JSON file
with open("CNN-Retro.json", "r") as json_file:
    loaded_model_json = json_file.read()
loaded_model = model_from_json(loaded_model_json)

# 2. Load trained weights
loaded_model.load_weights("CNN-Retro_final_weights.h5")

# 3. Compile the loaded model (make sure to use the same loss function and optimizer as when you originally trained the model)
opt = Adam()  # Nadam(learning_rate=0.001)
loaded_model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

# 4. Use the loaded model to make predictions on test data
test_predictions = loaded_model.predict([OHE_pp_test, padded_ecfp_input_test])

# Assuming your test labels are one-hot encoded, you might need to convert predictions to labels
test_labels_pred = np.argmax(test_predictions, axis=1)

# Calculate accuracy (you might need to adjust this depending on how your labels are formatted)
accuracy = np.mean(np.argmax(onehot_encoded_labels_test, axis=1) == test_labels_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")



# Calculate top 1% top 3% top 5% and top10% accuracy

In [ ]:
top1_accuracy = tf.keras.metrics.top_k_categorical_accuracy(onehot_encoded_labels_test, test_predictions, k=1)

# Evaluate the accuracy on your test data
print(f'Top-1 Accuracy: {np.mean(top1_accuracy.numpy())}')

In [ ]:
top3_accuracy = tf.keras.metrics.top_k_categorical_accuracy(onehot_encoded_labels_test, test_predictions, k=3)

# Evaluate the accuracy on your test data
print(f'Top-3 Accuracy: {np.mean(top3_accuracy.numpy())}')

In [ ]:
top5_accuracy = tf.keras.metrics.top_k_categorical_accuracy(onehot_encoded_labels_test, test_predictions, k=5)

# Evaluate the accuracy on your test data
print(f'Top-5 Accuracy: {np.mean(top5_accuracy.numpy())}')

In [ ]:
top10_accuracy = tf.keras.metrics.top_k_categorical_accuracy(onehot_encoded_labels_test, test_predictions, k=10)

# Evaluate the accuracy on your test data
print(f'Top-10 Accuracy: {np.mean(top10_accuracy.numpy())}')